In [30]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

# Add & investigating train and test data

## Train data

In [31]:
train_df = pd.read_csv("train_data.csv")

In [32]:
(train_df['sentiment'] == 0).sum()

np.int64(767059)

In [33]:
(train_df['sentiment'] == 1).sum()

np.int64(756916)

In [34]:
train_df['sentence'].str.len().quantile(0.95)

np.float64(123.0)

In [35]:
train_df.tail()

,sentence,sentiment
1523970,just woke up having no school is the best feel...,1
1523971,thewdb com very cool to hear old walt interviews,1
1523972,are you ready for your mojo makeover ask me fo...,1
1523973,happy th birthday to my boo of alll time tupac...,1
1523974,happy charitytuesday,1


In [36]:
train_df = train_df.sample(frac=1).reset_index(drop=True)
train_df.tail()

,sentence,sentiment
1523970,internet is down only thing thats working is t...,0
1523971,nature walk with the puppy,1
1523972,we did indeed see the brb last night the dance...,1
1523973,i thought so too sigh,0
1523974,can t wait gonna see it in d tomorrow,1


In [37]:
X_train = np.array(train_df["sentence"].values)
y_train = np.array(train_df["sentiment"].values)
X_train.shape, y_train.shape

((1523975,), (1523975,))

## Test data

In [38]:
test_df = pd.read_csv("test_data.csv")

In [39]:
test_df = test_df.sample(frac=1).reset_index(drop=True)
test_df.tail()

,sentence,sentiment
354,wow everyone at the google i o conference got ...,1
355,we love you too and don t want you to die late...,0
356,malcolm gladwell might be my new man crush,1
357,lol ah my skin is itchy damn lawnmowing,0
358,having the old coca cola guy on the gm board i...,0


In [40]:
X_test = np.array(test_df["sentence"].values)
y_test = np.array(test_df["sentiment"].values)
X_test.shape, y_test.shape

((359,), (359,))

# Tokenize

In [41]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

In [42]:
vocab_size = 16000
max_length = 123

In [43]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')

In [44]:
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

In [45]:
embedding_dim = 128

In [46]:
model = keras.Sequential([
        keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            input_length=max_length,
            name="embedding_5"
        ),
        keras.layers.Flatten(name="flatten_5"),
        keras.layers.Dense(64, activation="relu", name="dense_10"),
        keras.layers.Dense(1, activation="sigmoid", name="dense_11"),
    ])

model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
)

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [47]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [49]:
history = model.fit(
    X_train_pad, y_train,
    batch_size=32,
    epochs=3,
    validation_split=0.2,
    verbose=1
)

Epoch 1/3
38100/38100 ━━━━━━━━━━━━━━━━━━━━ 2178s 57ms/step - accuracy: 0.8167 - loss: 0.4025 - val_accuracy: 0.8052 - val_loss: 0.4232
Epoch 2/3
38100/38100 ━━━━━━━━━━━━━━━━━━━━ 2177s 57ms/step - accuracy: 0.8343 - loss: 0.3701 - val_accuracy: 0.8036 - val_loss: 0.4326
Epoch 3/3
38100/38100 ━━━━━━━━━━━━━━━━━━━━ 2208s 58ms/step - accuracy: 0.8536 - loss: 0.3334 - val_accuracy: 0.7984 - val_loss: 0.4630


In [51]:
loss, acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test Accuracy: {acc:.4f}")

Test Accuracy: 0.8245
